In [23]:
import requests


In [24]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

In [25]:
zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'

print(zip_url)

https://github.com/evidentlyai/docs/archive/refs/heads/main.zip


In [26]:
zip_response = requests.get(zip_url)

In [27]:
zip_response.status_code

200

In [28]:
import io
import zipfile

In [29]:
zip_archive = zipfile.ZipFile(
    io.BytesIO(zip_response.content)
)

In [30]:
zip_archive.namelist()[:10]

['docs-main/',
 'docs-main/api-reference/',
 'docs-main/api-reference/endpoint/',
 'docs-main/api-reference/endpoint/create.mdx',
 'docs-main/api-reference/endpoint/delete.mdx',
 'docs-main/api-reference/endpoint/get.mdx',
 'docs-main/api-reference/introduction.mdx',
 'docs-main/api-reference/openapi.json',
 'docs-main/changelog/',
 'docs-main/changelog/changelog.mdx']

In [31]:
mdx_file = zip_archive.open(
    'docs-main/api-reference/endpoint/create.mdx'
)

In [32]:
mdx_content = mdx_file.read().decode('utf-8')

In [33]:
print(mdx_content[:500])

---
title: 'Create Plant'
openapi: 'POST /plants'
---



In [34]:
import frontmatter

In [35]:
post = frontmatter.loads(mdx_content)

In [36]:
post.metadata

{'title': 'Create Plant', 'openapi': 'POST /plants'}

In [37]:
post.content

''

In [38]:
filenames = zip_archive.namelist()

In [39]:
len(filenames)

278

In [40]:
for filename in filenames:
    if filename.endswith(('.md', '.mdx')):
        print(filename)

docs-main/api-reference/endpoint/create.mdx
docs-main/api-reference/endpoint/delete.mdx
docs-main/api-reference/endpoint/get.mdx
docs-main/api-reference/introduction.mdx
docs-main/changelog/changelog.mdx
docs-main/docs/library/data_definition.mdx
docs-main/docs/library/descriptors.mdx
docs-main/docs/library/evaluations_overview.mdx
docs-main/docs/library/leftover_content.mdx
docs-main/docs/library/metric_generator.mdx
docs-main/docs/library/output_formats.mdx
docs-main/docs/library/overview.mdx
docs-main/docs/library/prompt_optimization.mdx
docs-main/docs/library/report.mdx
docs-main/docs/library/synthetic_data_api.mdx
docs-main/docs/library/tags_metadata.mdx
docs-main/docs/library/tests.mdx
docs-main/docs/platform/alerts.mdx
docs-main/docs/platform/dashboard_add_panels.mdx
docs-main/docs/platform/dashboard_add_panels_ui.mdx
docs-main/docs/platform/dashboard_overview.mdx
docs-main/docs/platform/dashboard_panel_types.mdx
docs-main/docs/platform/datasets_generate.mdx
docs-main/docs/platf

In [41]:
documents = []

for filename in filenames:
    if filename.endswith(('.md', '.mdx')):
        file = zip_archive.open(filename)
        content = file.read().decode('utf-8')

        post = frontmatter.loads(content)

        document = {
            "filename": filename,
            "metadata": post.metadata,
            "content": post.content
        }
        documents.append(document)
        

In [42]:
len(documents)

95

In [43]:
documents[0]["content"][:500]

''

In [44]:
documents[0]

{'filename': 'docs-main/api-reference/endpoint/create.mdx',
 'metadata': {'title': 'Create Plant', 'openapi': 'POST /plants'},
 'content': ''}

In [45]:
for document in documents:
    if document["content"]:
        print(document["filename"])
        print(document["content"][:500])
        break

docs-main/api-reference/introduction.mdx
<Note>
  If you're not looking to build API reference documentation, you can delete
  this section by removing the api-reference folder.
</Note>

## Welcome

There are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.

<Card
  title="Plant Store Endpoints"
  icon="leaf"
  href="https://github.


In [46]:
documents[1]

{'filename': 'docs-main/api-reference/endpoint/delete.mdx',
 'metadata': {'title': 'Delete Plant', 'openapi': 'DELETE /plants/{id}'},
 'content': ''}

In [47]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [48]:
files[0]

RawRepositoryFile(filename='api-reference/endpoint/create.mdx', content="---\ntitle: 'Create Plant'\nopenapi: 'POST /plants'\n---")

In [49]:
documents = []

for file in files:
    document = file.parse()
    documents.append(document)

In [50]:
len(documents)

95

In [51]:
from minsearch import Index

In [52]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

In [53]:
index.fit(documents)

In [54]:
query = "LLM as a Judge"

In [55]:
results = index.search(query, num_results=5)


In [56]:
len(results)

5

In [57]:
results[0]

{'title': 'LLM as a judge',
 'description': 'How to create and evaluate an LLM judge.',
 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create

In [58]:
results[1]

{'title': 'LLM-as-a-jury',
 'description': 'Evaluate the LLM outputs with multiple LLMs.',
 'content': 'This evaluation approach uses multiple LLMs to evaluate the same output. You can do this to obtain an aggregate evaluation result — e.g., consider an output a "pass" only if all or the majority of LLMs approve — or to explicitly surface disagreements.\n\nBlog explaining the concept of LLM jury: https://www.evidentlyai.com/blog/llm-judges-jury .\n\nCode example as a Jupyter notebook: https://github.com/evidentlyai/community-examples/blob/main/tutorials/LLM_as_a_jury_Example.ipynb\n\n## Preparation\n\nInstall Evidently:\n\n```python\npip install evidently litellm \n```\n\n(Or install `evidently[llm]`.)\n\nImport the components you\'ll use:\n\n```python\nimport pandas as pd\nfrom evidently import Dataset\nfrom evidently import DataDefinition\nfrom evidently import Report\nfrom evidently.presets import TextEvals\nfrom evidently.tests import eq, is_in, not_in\nfrom evidently.descriptors i

In [59]:
from gitsource import chunk_documents

In [60]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [61]:
len(document_chunks)

382

In [62]:
document_chunks[0]

{'start': 0,
 'content': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'api-reference/introduction.mdx'}

In [63]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

In [64]:
chunk_index.fit(document_chunks)

In [65]:
results = chunk_index.search(query, num_results=5)

In [66]:
results[0]

{'start': 0,
 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

In [67]:
import json

In [68]:
search_result_json = json.dumps(results, indent=2)

In [69]:
instructions = """You're a course assistant, your task is to answer the QUESTION from the course students using the provided CONTEXT"""

In [70]:
user_prompt = f"<QUESTION>{query}</QUESTION><CONTEXT>{search_result_json}</CONTEXT>"

In [71]:
answer = llm(user_prompt, instructions)

NameError: name 'llm' is not defined

In [75]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []
    if instructions is not None:
        messages.append({"role": "system", "content": instructions})
    messages.append({"role": "user", "content": user_prompt})
    response = openai_client.responses.create(model=model, input=messages)
    return response.output_text

In [78]:
from openai import OpenAI
openai_client = OpenAI()

In [79]:
answer = llm(user_prompt, instructions)

In [80]:
answer

'### LLM as a Judge\n\nIn this tutorial, you\'ll learn how to evaluate text using a Large Language Model (LLM) as a judge by following these key points:\n\n1. **Evaluation Methods**:\n   - **Reference-based**: Compare new responses to an existing "ground truth" response.\n   - **Open-ended**: Use custom criteria to evaluate responses when no reference is available.\n\n2. **Tutorial Overview**:\n   - **Create an Evaluation Dataset**: Formulate a simple Q&A dataset containing questions, target responses, new responses, and manual labels.\n   - **Design an LLM Evaluator Prompt**: Create prompts for evaluating correctness and verbosity.\n   - **Evaluate the Judge**: Compare the evaluations made by the LLM with manual labels to assess its performance.\n\n3. **Requirements**:\n   - Basic Python knowledge.\n   - An OpenAI API key for the LLM evaluator.\n\n4. **Installation and Setup**:\n   - Install the required Python library using `pip install evidently`.\n   - Import necessary modules and 

In [81]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [82]:
rag("how do I implement llm as a judge?")

NameError: name 'search' is not defined

In [83]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [85]:
def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)
    user_prompt = f"<QUESTION>{query}</QUESTION><CONTEXT>{search_result_json}</CONTEXT>"
    return user_prompt

In [86]:
rag("how do I implement llm as a judge?")

'To implement an LLM as a judge, follow these steps:\n\n1. **Set Up Your Environment**:\n   - Ensure you have basic Python knowledge and install the required package:\n     ```python\n     pip install evidently\n     ```\n\n2. **Import Required Libraries**:\n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset, DataDefinition, Report, BinaryClassification\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n3. **Configure Your OpenAI API Key**:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n4. **Create an Evaluation Dataset**:\n   Design a dataset that includes questions, target responses, new responses, and manual labels to determine correctness. For example:\n   ```python\n   data = [\n       ["Hi there, how do I reset my password?", "Target response text...", "New response text...", "incorrect", "adds new information"],\n       ...\n   ]\n   eval_dataset = pd.DataFrame(dat